In [1]:
import tensorflow as tf
import numpy as np
import os

from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model

2026-04-15 06:32:37.499851: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 06:32:37.499912: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 06:32:37.501140: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 06:32:37.508237: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-15 06:32:38.131177: W tensorflow/compiler/tf2

In [2]:
DATASET_PATH = "/mnt/d/Mtool/archive/train_data"

IMG_SIZE = (224,224)
BATCH_SIZE = 32

In [3]:
datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='training'
)

val_data = datagen.flow_from_directory(
    DATASET_PATH,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    subset='validation'
)

Found 53217 images belonging to 2 classes.
Found 13304 images belonging to 2 classes.


In [4]:
base_model = MobileNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224,224,3)
)

base_model.trainable = False

2026-04-15 06:32:39.428487: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 06:32:39.477411: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 06:32:39.477470: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 06:32:39.479183: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 06:32:39.479232: I external/local_xla/xla/stream_executor

In [5]:
x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128,activation='relu')(x)
predictions = Dense(1,activation='sigmoid')(x)

model = Model(inputs=base_model.input,outputs=predictions)

In [6]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "model"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_1 (InputLayer)        [(None, 224, 224, 3)]        0         []                            
                                                                                                  
 Conv1 (Conv2D)              (None, 112, 112, 32)         864       ['input_1[0][0]']             
                                                                                                  
 bn_Conv1 (BatchNormalizati  (None, 112, 112, 32)         128       ['Conv1[0][0]']               
 on)                                                                                              
                                                                                                  
 Conv1_relu (ReLU)           (None, 112, 112, 32)         0         ['bn_Conv1[0][0]']        

In [7]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10


2026-04-15 06:32:43.110737: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8902
2026-04-15 06:32:45.940936: I external/local_xla/xla/service/service.cc:168] XLA service 0x7287b4fe8040 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2026-04-15 06:32:45.940984: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce RTX 3060 Laptop GPU, Compute Capability 8.6
2026-04-15 06:32:45.954608: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1776227566.047542    1315 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1664/1664 [==============================] - 383s 228ms/step - loss: 0.1980 - accuracy: 0.9145 - val_loss: 2.9167 - val_accuracy: 0.5672
Epoch 2/10
1664/1664 [==============================] - 415s 249ms/step - loss: 0.1105 - accuracy: 0.9555 - val_loss: 3.3038 - val_accuracy: 0.5427
Epoch 3/10
1664/1664 [==============================] - 421s 253ms/step - loss: 0.0881 - accuracy: 0.9643 - val_loss: 3.1717 - val_accuracy: 0.6012
Epoch 4/10
1664/1664 [==============================] - 463s 278ms/step - loss: 0.0714 - accuracy: 0.9713 - val_loss: 2.7741 - val_accuracy: 0.6136
Epoch 5/10
1664/1664 [==============================] - 443s 266ms/step - loss: 0.0628 - accuracy: 0.9749 - val_loss: 3.2056 - val_accuracy: 0.5965
Epoch 6/10
1664/1664 [==============================] - 422s 253ms/step - loss: 0.0560 - accuracy: 0.9779 - val_loss: 3.3837 - val_accuracy: 0.5925
Epoch 7/10
1664/1664 [==============================] - 2114s 1s/step - loss: 0.0490 - accuracy: 0.9815 - val_loss: 2.8892 

In [9]:
import tensorflow as tf
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [10]:
model.save("drowsy_model.h5")

print("Model saved successfully")

/home/aly/miniconda3/envs/tf_gpu/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


Model saved successfully


In [11]:
img, label = next(val_data)

pred = model.predict(img)

print(pred[:5])

1/1 [==============================] - 1s 799ms/step
[[9.9999905e-01]
 [6.9711503e-04]
 [1.0000000e+00]
 [8.2519412e-01]
 [2.5111511e-01]]


In [2]:
import cv2
import mediapipe as mp
import numpy as np
import tensorflow as tf
import pygame

2026-04-15 08:28:45.417726: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-04-15 08:28:45.417810: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-04-15 08:28:45.419096: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-04-15 08:28:45.427084: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-04-15 08:28:46.387725: W tensorflow/compiler/tf2

pygame 2.6.1 (SDL 2.28.4, Python 3.10.20)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [3]:
model = tf.keras.models.load_model("drowsy_model.h5")

2026-04-15 08:28:47.580497: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 08:28:47.648132: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 08:28:47.648196: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 08:28:47.650661: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2026-04-15 08:28:47.650725: I external/local_xla/xla/stream_executor

In [4]:
mp_face_mesh = mp.solutions.face_mesh

face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=True
)

libEGL warning: DRI3 error: Could not get DRI3 device
libEGL warning: Ensure your X server supports DRI3 to get accelerated rendering
I0000 00:00:1776234529.077217    3550 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1776234529.093164    3731 gl_context.cc:344] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.2.8-0ubuntu0.24.04.1), renderer: llvmpipe (LLVM 20.1.2, 256 bits)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.


In [5]:
LEFT_EYE = [33,160,158,133,153,144]
RIGHT_EYE = [362,385,387,263,373,380]
MOUTH = [13,14,78,308]

In [6]:
def eye_aspect_ratio(eye):

    A = np.linalg.norm(eye[1]-eye[5])
    B = np.linalg.norm(eye[2]-eye[4])
    C = np.linalg.norm(eye[0]-eye[3])

    return (A+B)/(2.0*C)

In [7]:
def mouth_aspect_ratio(mouth):

    A = np.linalg.norm(mouth[0]-mouth[1])
    B = np.linalg.norm(mouth[2]-mouth[3])

    return A/B

In [8]:
pygame.mixer.init()

def alarm():
    pygame.mixer.music.load("alarm.wav")
    pygame.mixer.music.play()

In [11]:
EAR_THRESHOLD = 0.20
EAR_FRAMES = 30

MAR_THRESHOLD = 0.6
YAWN_FRAMES = 15

counter_eye = 0
counter_yawn = 0

In [14]:
ear_history = []

cap = cv2.VideoCapture("/mnt/d/Mtool/driver.mp4")

if not cap.isOpened():
    print("ERROR: Could not open video")

frame_count = 0
frame_skip = 3
cnn_result = False

while True:

    ret, frame = cap.read()
    if not ret:
        break

    frame_count += 1

    # Resize for speed
    frame = cv2.resize(frame,(320,240))

    # Frame skipping
    if frame_count % frame_skip != 0:
        cv2.imshow("Driver Monitoring System",frame)
        if cv2.waitKey(25) & 0xFF == ord("q"):
            break
        continue

    rgb = cv2.cvtColor(frame,cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)

    if results.multi_face_landmarks:

        for face_landmarks in results.multi_face_landmarks:

            h,w,_ = frame.shape

            left_eye=[]
            right_eye=[]
            mouth=[]

            for id in LEFT_EYE:
                x=int(face_landmarks.landmark[id].x*w)
                y=int(face_landmarks.landmark[id].y*h)
                left_eye.append([x,y])

            for id in RIGHT_EYE:
                x=int(face_landmarks.landmark[id].x*w)
                y=int(face_landmarks.landmark[id].y*h)
                right_eye.append([x,y])

            for id in MOUTH:
                x=int(face_landmarks.landmark[id].x*w)
                y=int(face_landmarks.landmark[id].y*h)
                mouth.append([x,y])

            left_eye=np.array(left_eye)
            right_eye=np.array(right_eye)
            mouth=np.array(mouth)

            # ---------------- EAR ----------------

            leftEAR=eye_aspect_ratio(left_eye)
            rightEAR=eye_aspect_ratio(right_eye)

            ear=(leftEAR+rightEAR)/2.0

            # EAR smoothing
            ear_history.append(ear)

            if len(ear_history)>7:
                ear_history.pop(0)

            ear=np.mean(ear_history)

            # ---------------- MAR ----------------

            mar=mouth_aspect_ratio(mouth)

            # ---------------- CNN (rarely) ----------------

            if frame_count % 25 == 0:

                face_img=cv2.resize(frame,(224,224))
                face_img=face_img/255.0
                face_img=np.expand_dims(face_img,axis=0)

                pred=model.predict(face_img,verbose=0)
                cnn_result=pred>0.65

            cnn_drowsy=cnn_result

            # ---------------- EAR TEMPORAL ----------------

            if ear < 0.19:
                counter_eye += 1
            else:
                counter_eye = max(0,counter_eye-3)

            ear_drowsy = counter_eye > 35

            # ---------------- YAWN TEMPORAL ----------------

            if mar > 0.65:
                counter_yawn += 1
            else:
                counter_yawn = max(0,counter_yawn-2)

            yawn_detected = counter_yawn > 20

            # ---------------- FINAL LOGIC ----------------

            drowsy = ear_drowsy or yawn_detected or (cnn_drowsy and ear < 0.23)

            if drowsy:

                cv2.putText(frame,"DROWSY",
                            (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,(0,0,255),3)

            else:

                cv2.putText(frame,"ALERT",
                            (20,40),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            1,(0,255,0),3)

            # ---------------- DEBUG INFO ----------------

            cv2.putText(frame,f"EAR:{ear:.2f}",
                        (20,80),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,(255,255,0),2)

            cv2.putText(frame,f"MAR:{mar:.2f}",
                        (20,110),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.7,(255,255,0),2)

            cv2.putText(frame,f"EYE CNT:{counter_eye}",
                        (20,140),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6,(255,255,255),1)

    cv2.imshow("Driver Monitoring System",frame)

    if cv2.waitKey(25) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()